In [ ]:
print("Structured Output")

In [4]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# LangChain | Structured Output 
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

from langchain_nebius import ChatNebius
from pydantic import BaseModel, Field
from typing_extensions import TypedDict, Annotated
from dotenv import load_dotenv

load_dotenv()

# ── model ─────────────────────────────────────────────────────────────
# model = ChatOpenRouter(model="z-ai/glm-5.1",temperature=0.7)
model = ChatNebius(model="MiniMaxAI/MiniMax-M2.5")


In [5]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# SECTION 1 · PYDANTIC — Social Media Post Agent
# Agent task: analyse a topic and produce a structured social post plan
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class Hashtag(BaseModel):
    tag: str = Field(description="Hashtag without the # symbol")
    relevance_score: float = Field(description="Relevance score 0.0 to 1.0")

class SocialMediaPost(BaseModel):
    """A structured social media post plan generated by an agent."""
    platform: str          = Field(description="Target platform: twitter, linkedin, instagram")
    caption: str           = Field(description="The post caption body")
    call_to_action: str    = Field(description="CTA line at the end of the post")
    hashtags: list[Hashtag]= Field(description="List of relevant hashtags with scores")
    best_time_to_post: str = Field(description="Recommended posting time e.g. '9am EST Tuesday'")
    estimated_reach: int   = Field(description="Estimated organic reach in number of accounts")

In [6]:
# 1a · Basic invoke — get a structured post plan
model_social = model.with_structured_output(SocialMediaPost)

raw_response = model_social.invoke(
    "Create a LinkedIn post plan for launching an open-source AI coding assistant tool."
)
response: SocialMediaPost = SocialMediaPost.model_validate(raw_response)

print(f"\n{'━'*60}")
print(f"  Social Media Agent — Structured Output (Pydantic)")
print(f"{'━'*60}")
print(f"Platform        : {response.platform}")
print(f"Caption         : {response.caption}")
print(f"CTA             : {response.call_to_action}")
print(f"Best Time       : {response.best_time_to_post}")
print(f"Est. Reach      : {response.estimated_reach:,}")
print(f"Hashtags        :")
for h in response.hashtags:
    print(f"  #{h.tag:<25} score={h.relevance_score}")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Social Media Agent — Structured Output (Pydantic)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Platform        : linkedin
Caption         : 🚀 Big news for developers! We're thrilled to announce the launch of [Tool Name], an open-source AI coding assistant designed to supercharge your development workflow.

🤖 What makes [Tool Name] different?
• Context-aware code suggestions that understand your entire codebase
• Multi-language support for Python, JavaScript, TypeScript, Go, Rust, and more
• Privacy-first: Your code never leaves your machine
• Fully customizable and extensible for your specific needs
• Open-source means you control your tools

💡 Why we built this:
We believe developers should have full control over their AI tools without sacrificing privacy or being locked into proprietary systems. [Tool Name] is our contribution to the developer community.

🔧 Try it today:
git clone [repo-url]

Join 500+ de

In [7]:
# 1b · include_raw=True — get parsed + raw AIMessage (for token usage etc.)
model_social_raw = model.with_structured_output(SocialMediaPost, include_raw=True)

response_raw = model_social_raw.invoke(
    "Create a Twitter post plan for a new Python debugging library launch."
)

print(f"\n{'━'*60}")
print(f"  include_raw=True — Parsed + Metadata")
print(f"{'━'*60}")
print(f"Parsed          : {response_raw['parsed']}")
print(f"Parsing Error   : {response_raw['parsing_error']}")
print(f"Token Usage     : {response_raw['raw'].usage_metadata}")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  include_raw=True — Parsed + Metadata
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Parsed          : platform='twitter' caption="Meet PyDeBug 🔍 — the Python debugging library you didn't know you needed! Get instant stack trace analysis, intelligent breakpoint suggestions, and visual error tracking. Stop guessing, start fixing. #PythonDev #Debugging #Programming" call_to_action='Try PyDeBug today and see your bugs in a whole new light!' hashtags=[Hashtag(tag='Python', relevance_score=0.95), Hashtag(tag='Debugging', relevance_score=0.95), Hashtag(tag='Programming', relevance_score=0.9), Hashtag(tag='DevTools', relevance_score=0.85), Hashtag(tag='OpenSource', relevance_score=0.8), Hashtag(tag='PythonDev', relevance_score=0.9)] best_time_to_post='10am EST Tuesday' estimated_reach=5000
Parsing Error   : None
Token Usage     : {'input_tokens': 431, 'output_tokens': 393, 'total_tokens': 824, 'input_token_details': 

In [9]:
# 1c · Nested Pydantic — Coding Agent PR Review Output
class FileReview(BaseModel):
    filename: str
    issues_found: int
    severity: str = Field(description="low | medium | high | critical")
    suggestion: str

class PRReviewReport(BaseModel):
    """Structured PR review report produced by a coding agent."""
    pr_title: str
    overall_score: float          = Field(description="Code quality score 0.0 to 10.0")
    approved: bool
    files_reviewed: list[FileReview]
    summary: str
    next_steps: list[str]

model_pr = model.with_structured_output(PRReviewReport)

raw_response = model_pr.invoke(
    """Review this PR: 'Add async rate limiter middleware to FastAPI app'. 
    It modifies: main.py, middleware/rate_limiter.py, tests/test_rate_limiter.py"""
)
response: PRReviewReport = PRReviewReport.model_validate(raw_response)

print(f"\n{'━'*60}")
print(f"  Coding Agent — Nested Pydantic PR Review")
print(f"{'━'*60}")
print(f"PR              : {response.pr_title}")
print(f"Score           : {response.overall_score}/10  |  Approved: {response.approved}")
print(f"Summary         : {response.summary}")
print(f"Files Reviewed  :")
for f in response.files_reviewed:
    print(f"  {f.filename:<40} [{f.severity}] — {f.suggestion}")
print(f"Next Steps      :")
for step in response.next_steps:
    print(f"  • {step}")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Coding Agent — Nested Pydantic PR Review
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
PR              : Add async rate limiter middleware to FastAPI app
Score           : 0.0/10  |  Approved: False
Summary         : I cannot review this PR without seeing the actual code changes. The files mentioned (main.py, middleware/rate_limiter.py, tests/test_rate_limiter.py) were not provided. Please share the code diff or the contents of these modified files so I can perform a thorough review.
Files Reviewed  :
Next Steps      :
  • Provide the code diff or the contents of main.py, middleware/rate_limiter.py, and tests/test_rate_limiter.py
  • Alternatively, share a link to the PR if working within a git hosting platform integration


In [10]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# SECTION 2 · TYPEDDICT — Coding Agent Bug Triage
# Simpler than Pydantic, no runtime validation — good for pipelines
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class StackFrame(TypedDict):
    file: str
    line: int
    function: str

class BugReport(TypedDict):
    """Structured bug triage output from a coding agent."""
    bug_id:       Annotated[str,        ..., "Unique bug identifier e.g. BUG-042"]
    title:        Annotated[str,        ..., "Short bug title"]
    severity:     Annotated[str,        ..., "low | medium | high | critical"]
    root_cause:   Annotated[str,        ..., "One-line root cause description"]
    stack_trace:  Annotated[list[StackFrame], ..., "Relevant stack frames"]
    fix_estimate: Annotated[str,        ..., "Time estimate e.g. '2 hours'"]
    assignee:     Annotated[str | None, ..., "GitHub username or null if unassigned"]

In [11]:
model_bug = model.with_structured_output(BugReport)

response = model_bug.invoke(
    """Triage this error: 
    'KeyError: user_id in session_manager.py line 87 inside get_active_session(). 
     Occurs when user logs in from two devices simultaneously. 
     Affects ~3% of users in production.'"""
)

print(f"\n{'━'*60}")
print(f"  Coding Agent — TypedDict Bug Triage")
print(f"{'━'*60}")
print(f"Bug ID          : {response['bug_id']}")
print(f"Title           : {response['title']}")
print(f"Severity        : {response['severity']}")
print(f"Root Cause      : {response['root_cause']}")
print(f"Fix Estimate    : {response['fix_estimate']}")
print(f"Assignee        : {response['assignee']}")
print(f"Stack Frames    :")
for frame in response['stack_trace']:
    print(f"  {frame['file']}:{frame['line']} in {frame['function']}()")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Coding Agent — TypedDict Bug Triage
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Bug ID          : BUG-001
Title           : KeyError: user_id in session_manager.py get_active_session() with simultaneous device logins
Severity        : high
Root Cause      : Race condition in session management when user logs in from multiple devices simultaneously - get_active_session() assumes user_id exists in session data but doesn't handle concurrent session creation properly
Fix Estimate    : 4 hours
Assignee        : null
Stack Frames    :
  session_manager.py:87 in get_active_session()


In [12]:
# Nested TypedDict — Social Media Agent Campaign Planner
class PlatformSchedule(TypedDict):
    platform: str
    post_times: list[str]
    content_type: str

class ContentCalendar(TypedDict):
    """Weekly content calendar produced by a social media agent."""
    campaign_name:   Annotated[str,                    ..., "Name of the campaign"]
    brand_voice:     Annotated[str,                    ..., "e.g. professional, playful, bold"]
    weekly_schedule: Annotated[list[PlatformSchedule], ..., "Per-platform posting schedule"]
    kpi_targets:     Annotated[dict[str, int],         ..., "e.g. {'impressions': 50000}"]
    budget_usd:      Annotated[float | None,           ..., "Ad budget in USD or null"]

model_calendar = model.with_structured_output(ContentCalendar)

response = model_calendar.invoke(
    "Plan a 1-week social media campaign for a new AI developer tool targeting Python devs."
)

print(f"\n{'━'*60}")
print(f"  Social Media Agent — TypedDict Content Calendar")
print(f"{'━'*60}")
print(f"Campaign        : {response['campaign_name']}")
print(f"Brand Voice     : {response['brand_voice']}")
print(f"Budget          : ${response['budget_usd']}")
print(f"KPI Targets     : {response['kpi_targets']}")
print(f"Schedule        :")
for s in response['weekly_schedule']:
    print(f"  {s['platform']:<12} | {s['content_type']:<20} | {', '.join(s['post_times'])}")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Social Media Agent — TypedDict Content Calendar
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Campaign        : AI Developer Tool Launch Week
Brand Voice     : bold
Budget          : $5000.0
KPI Targets     : {'impressions': 150000, 'engagement_rate': 5, 'website_visits': 10000, 'signups': 2500}
Schedule        :
  Twitter/X    | announcement, feature highlight, developer tips, testimonial | 2024-01-15T09:00:00Z, 2024-01-17T14:00:00Z, 2024-01-19T10:00:00Z, 2024-01-21T09:00:00Z
  LinkedIn     | announcement, case study, thought leadership | 2024-01-15T10:00:00Z, 2024-01-18T12:00:00Z, 2024-01-20T10:00:00Z
  Hacker News  | launch story, technical deep-dive | 2024-01-15T11:00:00Z, 2024-01-19T11:00:00Z
  Reddit       | AMA, product demo    | 2024-01-16T15:00:00Z, 2024-01-20T15:00:00Z
  YouTube      | demo video, tutorial | 2024-01-14T12:00:00Z, 2024-01-21T12:00:00Z


In [25]:
from langchain_nebius import ChatNebius
from langchain_cerebras import ChatCerebras
from langchain_groq import ChatGroq
from pydantic import BaseModel, Field
from typing_extensions import TypedDict, Annotated
from dotenv import load_dotenv

load_dotenv()

model = ChatGroq(model="openai/gpt-oss-120b")

In [29]:
from collections.abc import Mapping
from pydantic import BaseModel

# Social Media Agent — Sentiment + Engagement Analysis (JSON Schema)
sentiment_schema = {
    "title": "PostAnalysis",
    "description": "Social media post sentiment and engagement analysis by an agent",
    "type": "object",
    "properties": {
        "sentiment": {"type": "string", "enum": ["positive", "neutral", "negative"]},
        "sentiment_score": {"type": "number", "description": "Score -1.0 to 1.0"},
        "toxicity_flag": {"type": "boolean", "description": "True if content is toxic"},
        "topics": {"type": "array", "items": {"type": "string"}},
        "predicted_engagement": {
            "type": "object",
            "properties": {
                "likes": {"type": "integer"},
                "shares": {"type": "integer"},
                "comments": {"type": "integer"},
            },
            "required": ["likes", "shares", "comments"],
        },
        "improvement_tips": {"type": "array", "items": {"type": "string"}, "maxItems": 3},
    },
    "required": [
        "sentiment",
        "sentiment_score",
        "toxicity_flag",
        "topics",
        "predicted_engagement",
        "improvement_tips",
    ],
}

model_sentiment = model.with_structured_output(sentiment_schema, method="json_schema")

response = model_sentiment.invoke(
    """Analyse this post:
    'Just shipped v2.0 of our open-source LLM router — 10x faster,
     half the cost. Try it now and let us know what you think! 🚀 #AI #OpenSource'"""
)

if isinstance(response, BaseModel):
    analysis = response.model_dump()
elif isinstance(response, Mapping):
    analysis = dict(response)
else:
    raise TypeError(f"Unexpected structured output type: {type(response)!r}")

print(f"\n{'━' * 60}")
print("  Social Media Agent — JSON Schema Sentiment Analysis")
print(f"{'━' * 60}")
print(f"Sentiment       : {analysis['sentiment']}  (score={analysis['sentiment_score']})")
print(f"Toxicity Flag   : {analysis['toxicity_flag']}")
print(f"Topics          : {', '.join(analysis['topics'])}")
print("Predicted Engagement:")
e = analysis["predicted_engagement"]
print(f"  Likes={e['likes']}  Shares={e['shares']}  Comments={e['comments']}")
print("Improvement Tips:")
for tip in analysis["improvement_tips"]:
    print(f"  • {tip}")



━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Social Media Agent — JSON Schema Sentiment Analysis
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Sentiment       : positive  (score=0.82)
Toxicity Flag   : False
Topics          : LLM, OpenSource, AI, Performance, SoftwareRelease
Predicted Engagement:
  Likes=152  Shares=34  Comments=19
Improvement Tips:
  • Add a direct link to the repository for easy access
  • Include a brief demo video or GIF to showcase speed gains
  • Encourage community contributions with a clear call‑to‑action


In [33]:
import json
result = json.dumps(response, indent=2)
print(result)

{
  "sentiment": "positive",
  "sentiment_score": 0.82,
  "toxicity_flag": false,
  "topics": [
    "LLM",
    "OpenSource",
    "AI",
    "Performance",
    "SoftwareRelease"
  ],
  "predicted_engagement": {
    "likes": 152,
    "shares": 34,
    "comments": 19
  },
  "improvement_tips": [
    "Add a direct link to the repository for easy access",
    "Include a brief demo video or GIF to showcase speed gains",
    "Encourage community contributions with a clear call\u2011to\u2011action"
  ]
}


In [34]:
from collections.abc import Mapping
from pydantic import BaseModel

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# SECTION 3 · JSON SCHEMA — Maximum control, provider-agnostic
# Best for: APIs, cross-language pipelines, strict schema enforcement
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# Coding Agent — Code Review JSON Schema
code_review_schema = {
    "title": "CodeReview",
    "description": "Automated code review output from a coding agent",
    "type": "object",
    "properties": {
        "language": {"type": "string", "description": "Programming language detected"},
        "quality_score": {"type": "number", "description": "Overall quality 0.0 to 10.0"},
        "has_tests": {"type": "boolean", "description": "Whether tests are present"},
        "vulnerabilities": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "type": {"type": "string"},
                    "line": {"type": "integer"},
                    "description": {"type": "string"},
                    "cwe_id": {"type": "string", "description": "CWE identifier e.g. CWE-89"},
                },
                "required": ["type", "line", "description", "cwe_id"],
            },
        },
        "refactor_suggestions": {"type": "array", "items": {"type": "string"}},
        "auto_fixable": {"type": "boolean", "description": "Can issues be auto-fixed?"},
    },
    "required": [
        "language",
        "quality_score",
        "has_tests",
        "vulnerabilities",
        "refactor_suggestions",
        "auto_fixable",
    ],
}

model_review = model.with_structured_output(code_review_schema, method="json_mode")

print(f"\n{'━' * 60}")
print("  Coding Agent — JSON Schema Code Review")
print(f"{'━' * 60}")

response = model_review.invoke(
    """Review this Python FastAPI code change.
Return a JSON object that strictly includes these keys:
language, quality_score, has_tests, vulnerabilities, refactor_suggestions, auto_fixable."""
)

if isinstance(response, BaseModel):
    review = response.model_dump()
elif isinstance(response, Mapping):
    review = dict(response)
else:
    raise TypeError(f"Unexpected structured output type: {type(response)!r}")

print(f"Language        : {review.get('language', 'unknown')}")
print(f"Quality Score   : {review['quality_score']}/10")
print(f"Has Tests       : {review['has_tests']}")
print(f"Auto Fixable    : {review['auto_fixable']}")
print("Vulnerabilities :")
for v in review["vulnerabilities"]:
    print(f"  [{v['cwe_id']}] line {v['line']} — {v['type']}: {v['description']}")
print("Refactor Tips   :")
for tip in review["refactor_suggestions"]:
    print(f"  • {tip}")



━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Coding Agent — JSON Schema Code Review
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Language        : Python
Quality Score   : 0/10
Has Tests       : False
Auto Fixable    : False
Vulnerabilities :
Refactor Tips   :


In [36]:
import json
result = json.dumps(response, indent=2)
print(result)

{
  "language": "Python",
  "quality_score": 0,
  "has_tests": false,
  "vulnerabilities": [],
  "refactor_suggestions": [],
  "auto_fixable": false
}
